In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

pdf = spark.table("launch_reliability.gold.model_dataset").toPandas()

pdf["failed"] = pdf["failed"].astype(int)
pdf["decade"] = pdf["decade"].astype(str)
pdf["return_to_flight"] = pdf["return_to_flight"].astype(int)

print(f"{len(pdf):,} launches, {pdf['failed'].sum()} failures")

In [0]:
# Failure odds vs. log flight number, controlling for decade.
# Standard errors are clustered by rocket type: launches of the same rocket
# aren't independent, so treating them as independent would overstate certainty.
rq1 = smf.logit(
    "failed ~ log_vehicle_flight + C(decade, Treatment('1990'))",
    data=pdf,
).fit(cov_type="cluster", cov_kwds={"groups": pdf["lv_type"].astype("category").cat.codes},
      disp=False)

# Translate the slope into "effect of doubling flight experience"
b = rq1.params["log_vehicle_flight"]
lo, hi = rq1.conf_int().loc["log_vehicle_flight"]
print("RQ1: each doubling of a rocket's flight count multiplies the odds of failure by")
print(f"  {np.exp(b * np.log(2)):.3f}  (95% CI {np.exp(lo * np.log(2)):.3f} to "
      f"{np.exp(hi * np.log(2)):.3f}),  p = {rq1.pvalues['log_vehicle_flight']:.2g}")

rq1.summary()

In [0]:
# Interaction: lets the learning-curve slope differ by provider group
rq2 = smf.logit(
    "failed ~ log_vehicle_flight * C(provider_group, Treatment('government'))"
    " + C(decade, Treatment('1990'))",
    data=pdf,
).fit(cov_type="cluster", cov_kwds={"groups": pdf["lv_type"].astype("category").cat.codes},
      disp=False)

p = rq2.params
slope_gov = p["log_vehicle_flight"]
interaction = [k for k in p.index if k.startswith("log_vehicle_flight:")][0]
slope_com = slope_gov + p[interaction]

print("Odds multiplier per doubling of flight experience:")
print(f"  government: {np.exp(slope_gov * np.log(2)):.3f}")
print(f"  commercial: {np.exp(slope_com * np.log(2)):.3f}")
print(f"Difference in slopes (interaction): p = {rq2.pvalues[interaction]:.2g}")

rq2.summary()


In [0]:
flights = np.arange(1, 101)
fig, ax = plt.subplots(figsize=(7, 4))

for group, color in [("government", "#4C72B0"), ("commercial", "#DD8452")]:
    grid = pd.DataFrame({"log_vehicle_flight": np.log(flights),
                         "provider_group": group, "decade": "2000"})
    ax.plot(flights, rq2.predict(grid) * 100, color=color, label=group.capitalize())

ax.set_xscale("log")
ax.set_xlabel("Vehicle flight number (log scale)")
ax.set_ylabel("Predicted failure rate (%)")
ax.set_title("Model-estimated learning curves (2000s launches)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## RQ1: How does failure risk drop as a vehicle accumulates flights?

**What I did**
- Fit a logistic regression on all 7,407 launches: failure vs. log of vehicle flight number, controlling for decade
- Clustered standard errors by rocket type, since launches of the same rocket aren't independent

**Results**
- **Each doubling of a rocket's flight count cuts the odds of failure by 26%** (95% CI 21–31%, p < 0.001)
- Launches in the **1950s** and **1960s** had **10.6×** and **2.3×** the odds of failure of the 1990s
- From the 1970s on, decade had no significant effect once experience was accounted for

## RQ2: Do commercial providers mature faster than government programs?

**What I did**
- Added an interaction between flight experience and provider group (commercial vs. government), still controlling for decade with clustered standard errors

**Results**
- Government providers cut failure odds by **22%** per doubling of experience; commercial providers by **40%**
- The difference in learning rates is significant (**p < 0.001**), so commercial learning curves are steeper even after controlling for era
- Maiden flights were equally risky for both groups (p = 0.39): commercial providers start the same but improve faster
- Example (1990s launch): predicted failure rate falls from **16.6% to 5.3%** for government and from **19.9% to 1.9%** for commercial between flight 1 and flight 32